<a href="https://colab.research.google.com/github/Mafervv149/tp-grupal-datos-fce/blob/vani/Analisis_Natalidad_RENAPER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bloque 1: Carga de Datos

###  Carga y exploración inicial
#### a) Carga del dataset
En esta sección importamos la librería `pandas`,  y cargamos el dataset de nacimientos del RENAPER (2012-2024). Mostramos las primeras y últimas filas para verificar que la estructura se haya leído correctamente.

###  Carga y exploración inicial
#### a) Carga del dataset
En esta sección importamos la librería pandas, numpy y cargamos el dataset de nacimientos del RENAPER (2012-2024). Mostramos las primeras y últimas filas para verificar que la estructura se haya leído correctamente.

In [ ]:
import pandas as pd
import numpy as np




##### Usamos keep_default_na=False temporalmente para que no borre los "NA" de texto,
##### o lo dejamos por defecto para analizar la calidad de datos reales.texto en cursiva

In [ ]:
keep_default_na=False

In [ ]:
df = pd.read_csv('/nacimientos_renaper.csv')

In [ ]:
# Mostramos las primeras 5 filas
print("--- PRIMERAS FILAS (.head()) ---")
display(df.head())

--- PRIMERAS FILAS (.head()) ---


,provincia_nombre,provincia_id,grupo_etario_madre,anio,cantidad
0,Buenos Aires,6.0,10-14,2012,974
1,Buenos Aires,6.0,10-14,2013,992
2,Buenos Aires,6.0,10-14,2014,968
3,Buenos Aires,6.0,10-14,2015,848
4,Buenos Aires,6.0,10-14,2016,689


In [ ]:
# Mostramos las últimas 5 filas
print("\n--- ÚLTIMAS FILAS (.tail()) ---")
display(df.tail())


--- ÚLTIMAS FILAS (.tail()) ---


,provincia_nombre,provincia_id,grupo_etario_madre,anio,cantidad
2491,Tierra del Fuego,94.0,45-49,2020,2
2492,Tierra del Fuego,94.0,45-49,2021,2
2493,Tierra del Fuego,94.0,45-49,2022,3
2494,Tierra del Fuego,94.0,45-49,2023,10
2495,Tierra del Fuego,94.0,45-49,2024,4


# Bloque 2: Estructura del Dataset

b) Descripción de la estructura
Analizamos las dimensiones del DataFrame (filas y columnas), identificamos las variables disponibles y sus tipos de datos primitivos, determinando además el alcance temporal y geográfico de los registros.

In [ ]:
# Dimensiones
print(f"Dimensiones del dataset (Filas, Columnas): {df.shape}\n")


Dimensiones del dataset (Filas, Columnas): (2496, 5)



In [ ]:
# Tipos de datos y columnas
print("Tipos de datos por columna:")
print(df.dtypes)

Tipos de datos por columna:
provincia_nombre       object
provincia_id          float64
grupo_etario_madre     object
anio                    int64
cantidad                int64
dtype: object


In [ ]:
# Cobertura temporal
anio_min = df['anio'].min()
anio_max = df['anio'].max()
print(f"\nCoberura temporal: Desde el año {anio_min} hasta el {anio_max}")



Coberura temporal: Desde el año 2012 hasta el 2024


In [ ]:
# Cobertura geográfica (provincias únicas)
provincias = df['provincia_nombre'].unique()
print(f"Cobertura geográfica: {len(provincias)} jurisdicciones analizadas.")

Cobertura geográfica: 24 jurisdicciones analizadas.


# Bloque 3: Calidad de los Datos

c) Análisis de calidad y estadísticas descriptivas
Evaluamos la presencia de valores nulos o faltantes, la existencia de filas completamente duplicadas y obtenemos un resumen estadístico de las variables numéricas para detectar anomalías o desvíos en los datos de cantidad de nacimientos.

In [ ]:
# Valores nulos por columna
print("--- Valores Nulos por Columna ---")
print(df.isnull().sum())

--- Valores Nulos por Columna ---
provincia_nombre        0
provincia_id          520
grupo_etario_madre      0
anio                    0
cantidad                0
dtype: int64


In [ ]:
# Duplicados
duplicados = df.duplicated().sum()
print(f"\nCantidad de filas completamente duplicadas: {duplicados}")


Cantidad de filas completamente duplicadas: 0


In [ ]:
# Estadísticas descriptivas de la variable cuantitativa
print("\n--- Estadísticas Descriptivas de 'cantidad' ---")
display(df['cantidad'].describe())


--- Estadísticas Descriptivas de 'cantidad' ---


,cantidad
count,2496.000000
mean,3227.312099
std,7579.433314
min,0.000000
25%,188.000000
50%,1231.000000
75%,3264.750000
max,72291.000000


# Bloque 4: El "Hallazgo"

### 🔍 Comentarios sobre los Hallazgos Iniciales


1. **Estructura y Dimensiones:** El dataset posee un formato de panel perfectamente balanceado por provincias, grupos etarios de la madre (de 10-14 hasta 45-49 años) y años calendario. La cobertura abarca desde 2012 hasta 2024, permitiendo observar una serie de tiempo sólida de 13 años.
2. **El Problema de los Nulos camuflados (NA):** Al ejecutar `.isnull().sum()`, Python podría reportar que no hay nulos si lee "NA" como texto en provincias como Córdoba o Tucumán. Sin embargo, sabemos que "NA" en este caso representa la ausencia de la ID numérica de la provincia. Este será un punto crítico de limpieza en la fase de transformaciones.
3. **Distribución de la Variable Cantidad:** El análisis descriptivo de la columna `cantidad` revela una asimetría muy marcada. Mientras que el percentil 25% registra valores pequeños (alrededor de 20 a 30 nacimientos en provincias chicas o rangos etarios extremos), el valor máximo es de 72,291 nacimientos (que pertenece a la provincia de Buenos Aires en los grupos centrales). Esto demuestra que el volumen bruto está altamente condicionado por el peso demográfico de cada provincia, justificando la necesidad de usar *tasas o participaciones porcentuales* para realizar una comparación justa entre jurisdicciones.

#Transformaciones y macrocategorías de edad materna


## A partir de aca falta acomodar y ver bien todo ###

In [ ]:
# Definimos el diccionario de mapeo demográfico
mapeo_edades = {
    '10-14': 'Maternidad Temprana (<20)',
    '15-19': 'Maternidad Temprana (<20)',
    '20-24': 'Maternidad Plena (20-34)',
    '25-29': 'Maternidad Plena (20-34)',
    '30-34': 'Maternidad Plena (20-34)',
    '35-39': 'Maternidad Tardía (35+)',
    '40-44': 'Maternidad Tardía (35+)',
    '45-49': 'Maternidad Tardía (35+)'
}


Transformación 1: Creamos la columna derivada con las categorías macro


In [ ]:
df['macro_edad_madre'] = df['grupo_etario_madre'].map(mapeo_edades)

Transformación 2: Filtrado para el período de análisis profundo (2015-2024)


In [ ]:
df_periodo_estudio = df[df['anio'] >= 2015].copy()


Resumen Estadístico con .groupby()


# Resumen 1: Evolución del total absoluto de nacimientos por año


In [ ]:
nacimientos_por_anio = df.groupby('anio')['cantidad'].sum().reset_index()
print("\n--- Evolución del Total de Nacimientos en Argentina ---")
print(nacimientos_por_anio)



--- Evolución del Total de Nacimientos en Argentina ---
    anio  cantidad
0   2012    721076
1   2013    735954
2   2014    766809
3   2015    746633
4   2016    713615
5   2017    694147
6   2018    676093
7   2019    612928
8   2020    552060
9   2021    512380
10  2022    483206
11  2023    443538
12  2024    396932


# Resumen 2: Evolución de la participación de las macrocategorías de edad


In [ ]:
nacimientos_por_grupo = df.groupby(['anio', 'macro_edad_madre'])['cantidad'].sum().unstack()


Convertimos a porcentajes relativos por año para medir el impacto estructural

In [ ]:
participacion_por_grupo = nacimientos_por_grupo.div(nacimientos_por_grupo.sum(axis=1), axis=0) * 100


print("\n--- Participación Porcentual (%) por Grupo Etario Materno ---")
print(participacion_por_grupo.round(2))



--- Participación Porcentual (%) por Grupo Etario Materno ---
macro_edad_madre  Maternidad Plena (20-34)  Maternidad Tardía (35+)  \
anio                                                                  
2012                                 68.48                    14.70   
2013                                 68.39                    15.02   
2014                                 68.41                    15.35   
2015                                 68.37                    16.22   
2016                                 68.95                    16.69   
2017                                 69.19                    17.22   
2018                                 69.56                    17.59   
2019                                 69.97                    18.40   
2020                                 70.46                    19.15   
2021                                 70.79                    19.88   
2022                                 70.81                    20.15   
2023          